# 02 - PDF Extraction

## Objetivo
Este notebook sirve para inspeccionar y extraer información de los PDFs dentro de `data/raw/docs`.

## Alcance
Este notebook **no** reconstruye todavía el dataset final. Su propósito es:

- listar los PDFs disponibles,
- detectar cuáles tienen texto extraíble,
- extraer texto por página,
- intentar extraer tablas,
- exportar resultados a archivos `.txt` o `.csv`,
- documentar qué PDFs sí son útiles para el pipeline.

## Resultado esperado
Al final de este notebook deberías poder responder:

1. ¿Qué PDFs contienen texto real?
2. ¿Qué PDFs son escaneados o difíciles de extraer?
3. ¿Qué tablas o metadatos vale la pena incorporar al proyecto?

In [1]:
from pathlib import Path
import pandas as pd
import pdfplumber
import re

In [2]:
# Define la ruta del directorio de datos crudos
PDF_DIR = Path("../data/raw/docs")

pdf_files = list(PDF_DIR.glob("*.pdf"))

print("PDF encontrados:", len(pdf_files))

for f in pdf_files:
    print(f.name)

PDF encontrados: 69
Eventos de animales 1204.pdf
Eventos de animales 1213.pdf
Eventos de animales 1216.pdf
Eventos de animales 1225.pdf
Eventos de animales 1228.pdf
Eventos de animales 1235.pdf
Eventos de animales 1236.pdf
Eventos de animales 1242.pdf
Eventos de animales 1243.pdf
Eventos de animales 1497.pdf
Eventos de animales 1510.pdf
Eventos de animales 1555.pdf
Eventos de animales 1560.pdf
Eventos de animales 1567.pdf
Eventos de animales 1577.pdf
Eventos de animales 1590.pdf
Eventos de animales 1617.pdf
Eventos de animales 1619.pdf
Eventos de animales 1638.pdf
Eventos de animales 1644.pdf
Eventos de animales 2070.pdf
Eventos de animales 2074.pdf
Eventos de animales 2076.pdf
Eventos de animales 2082.pdf
Eventos de animales 2087.pdf
Eventos de animales 2108.pdf
Eventos de animales 2110.pdf
Eventos de animales 2119.pdf
Eventos de animales 2122...pdf
Eventos de animales 2150.pdf
Eventos de animales 2165.pdf
Eventos de animales 5767.pdf
Eventos de animales 5981.pdf
Eventos de animales 6

In [3]:
EVENT_KEYWORDS = [
    "Cambio ID",
    "Cambio ID Tran",
    "Cambio de grup",
    "Cambio tabla ali",
    "Condición corpo",
    "Secado",
    "Diagnósticos/Tr",
    "Control de Gest",
    "Invitación Visita",
    "Inseminación",
    "Celo",
    "Parto",
    "Peso",
    "Entrada",
]


def ExtractAnimalIdFromFilename(pdf_path):
    match = re.search(r"(\d+)", pdf_path.stem)
    return match.group(1) if match else None


def NormalizeLine(text):
    return re.sub(r"\s+", " ", text).strip()


def ExtractLactationMarker(line):
    match = re.search(r"N[ºo]\s*Lactaci[oó]n\s*(\d+)", line, re.IGNORECASE)
    return int(match.group(1)) if match else None


def LooksLikeEventStart(line):
    return any(line.startswith(keyword) for keyword in EVENT_KEYWORDS)

In [4]:
def ParsePdfEvents(pdf_path):

    animal_id = ExtractAnimalIdFromFilename(pdf_path)
    rows = []

    with pdfplumber.open(pdf_path) as pdf:

        current_lactation = None

        for page_idx, page in enumerate(pdf.pages, start=1):

            text = page.extract_text() or ""

            lines = [NormalizeLine(x) for x in text.splitlines() if NormalizeLine(x)]

            current_event = None

            for line in lines:

                lact = ExtractLactationMarker(line)
                if lact is not None:
                    current_lactation = lact
                    continue

                if LooksLikeEventStart(line):

                    if current_event is not None:
                        rows.append(current_event)

                    current_event = {
                        "animal_id": animal_id,
                        "page_number": page_idx,
                        "lactation_number": current_lactation,
                        "raw_event_text": line,
                        "source_file": pdf_path.name,
                    }

                else:
                    if current_event is not None:
                        current_event["raw_event_text"] += " " + line

            if current_event is not None:
                rows.append(current_event)

    return pd.DataFrame(rows)

In [5]:
test_pdf = pdf_files[0]

df_events = ParsePdfEvents(test_pdf)

df_events.head()

,animal_id,page_number,lactation_number,raw_event_text,source_file
0,1204,1,1,Cambio tabla ali 22/08/2 User1,Eventos de animales 1204.pdf
1,1204,1,1,Condición corporal: 3.25 - DDUP:,Eventos de animales 1204.pdf
2,1204,1,1,Condición corpo 22/08/2 473 User1 Dry Off SE S...,Eventos de animales 1204.pdf
3,1204,1,1,Secado 22/08/2 User1 GESTANTE EN LA 7 INSEM. S...,Eventos de animales 1204.pdf
4,1204,1,1,Cambio de grup 22/08/2 User1 Dns: VACUNA; Loc....,Eventos de animales 1204.pdf


In [6]:
all_events = []

for pdf_file in pdf_files:

    try:
        df = ParsePdfEvents(pdf_file)
        all_events.append(df)

    except Exception as e:
        print("Error en:", pdf_file.name, e)

events_df = pd.concat(all_events, ignore_index=True)

print("Eventos extraídos:", events_df.shape)

events_df.head()

Eventos extraídos: (6890, 5)


,animal_id,page_number,lactation_number,raw_event_text,source_file
0,1204,1,1,Cambio tabla ali 22/08/2 User1,Eventos de animales 1204.pdf
1,1204,1,1,Condición corporal: 3.25 - DDUP:,Eventos de animales 1204.pdf
2,1204,1,1,Condición corpo 22/08/2 473 User1 Dry Off SE S...,Eventos de animales 1204.pdf
3,1204,1,1,Secado 22/08/2 User1 GESTANTE EN LA 7 INSEM. S...,Eventos de animales 1204.pdf
4,1204,1,1,Cambio de grup 22/08/2 User1 Dns: VACUNA; Loc....,Eventos de animales 1204.pdf


In [7]:
events_df["event_date"] = events_df["raw_event_text"].str.extract(
    r"(\d{2}/\d{2}/\d{4})"
)

events_df["event_date"] = pd.to_datetime(
    events_df["event_date"],
    dayfirst=True,
    errors="coerce"
)

In [8]:
OUTPUT_PATH = Path("../data/interim/pdf_events.parquet")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

events_df.to_parquet(OUTPUT_PATH)

print("Dataset guardado en:", OUTPUT_PATH)

Dataset guardado en: ..\data\interim\pdf_events.parquet
